# 20 — Rewind

**What to do here:** edit [`nb20_config.py`](nb20_config.py), run all, open the chart, press
**⟲ Rewind** and click a candle.

---

The same chart as [notebook 19](19_fvg_viewer.ipynb) — same gaps, same levels, same sim, the
settings are a copy — with one thing added: **put the chart back to what was knowable at one bar.**

| | REWIND (the button) | AS_OF (the setting) |
|---|---|---|
| how | click a candle | set a timestamp, re-run |
| speed | instant | a re-run |
| what it does | clips what is drawn | the detectors never see the future |
| works in | the browser tab only | anywhere, including inline |
| use it for | looking around | pinning a study |

Both are honest, because every artefact here already carries its own birth: a gap has `born_i`,
a level a `start_i`, a trendline a `born_i`, a trade an entry. Rewinding is a **filter over what is
already true**, not a guess.

When you rewind you will see **a gap that was filled later still drawn open**, a trendline that
broke later drawn unbroken, and trades taken later gone. That is the point — it is what you knew.
Candles and EMAs are deliberately left running, because the reason to study a bar is to see what
happened next.

> Notebook 19 is untouched and behaves exactly as before.

## 1 · Settings

All of them live in [`nb20_config.py`](nb20_config.py). Edited it while the kernel was running?
**Re-run this cell** — it reloads.

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

# A kernel that imported `quant` before a feature existed keeps the OLD module in memory while
# Jupyter shows you the NEW file, so the traceback points at code that is not running.
from quant.viz import AS_OF_JS

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb20_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb20_config as CFG
importlib.reload(CFG)

print(f"{CFG.SYMBOL} {CFG.TF}  {CFG.VIEW_START} -> {CFG.AS_OF or CFG.END}"
      f"{'   (AS_OF pinned)' if CFG.AS_OF else ''}")
print(f"gaps from {CFG.TF}+{'+'.join(CFG.FVG_TFS)}   levels from {CFG.TF}+{'+'.join(CFG.SR_TFS)}")
print(f"rewind button: {'on' if CFG.REWIND else 'off'}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-10
gaps from 5m+15m+1h+4h   levels from 5m+15m+1h+4h
rewind button: on
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb20_config.py


## 2 · Run

One call, the same as notebook 19. If `AS_OF` is set the data simply stops there, so everything
below is point-in-time with no filtering at all.

In [2]:
from quant.research import fvg_lab

lab = fvg_lab(CFG)
lab.summary()

[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[04:22:29] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[04:22:33] INFO | quant.engine | backtest done | bars=11,521 trades=3 final_cash=9670.60 return=-3.29% dd=3.29% | 64.9 ms
[04:22:33] INFO | quant.engine | backtest done | bars=11,521 trades=5 final_cash=11015.55 return=10.16% dd=1.72% | 0.7 ms


 tf  gaps  open  median_size_atr  median_bars_to_fill
 5m  1128    62             0.78                  5.0
 4h    52     8             0.77                  6.0
 1h    55     6             0.82                  4.0
15m   108     7             0.80                  6.0

8 trades   +7.0R   50% win    (veto off)


## 3 · Chart

Open it in a browser tab — **the rewind button only exists there**, because it is injected into the
HTML alongside the plot.

Press **⟲ Rewind**, click any candle, and everything derived snaps back to that bar. **live** puts
it back. The dotted yellow line marks where you are.

In [3]:
fig = lab.figure()
# fig

In [4]:
from quant.viz import open_in_browser

open_in_browser(fig, "rewind.html", extra_js=AS_OF_JS)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/rewind.html')

---
## Appendix · what rewinding actually changes

With `REWIND = True` the S/R layer is drawn as **one segment per price revision** instead of one
line at its final level. That matters more than it sounds: a zone's level is nudged as later pivots
merge into it, so the single line the normal chart draws is **today's** level painted across the
whole window. Rewinding to a past bar then shows the level that was really in force there.

That is the only thing the flag changes about the drawing. Everything else — which gaps exist, when
they filled, where the lines are — was already causal.

In [5]:
lab.census()

,tf,gaps,open,median_size_atr,median_bars_to_fill
0,5m,1128,62,0.78,5.0
1,4h,52,8,0.77,6.0
2,1h,55,6,0.82,4.0
3,15m,108,7,0.80,6.0
